# spec

> Any documented HTTP API as a tool group: load a spec, read the operations, call one.


In [ ]:
#| default_exp spec

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from pathlib import Path
from fastcore.test import test_eq, test_fail

`LocalHost` can fetch a page and run a command, which means an agent asked to use an API
has two options: guess the request, or shell out to `curl` and guess it there. Both fail the
same way -- silently, with a 200 and the wrong body, or a 400 whose message is about a field
name nobody wrote down.

A specification removes the guessing. `fastspec` parses OpenAPI, Google discovery documents
and GraphQL introspection into `OpSpec` records -- verb, path, parameters, their types,
their defaults, their docs -- and `fastcore.apisurface` turns those into real Python
signatures. What the agent gets is not "make an HTTP request", it is `list_repo_issues(owner,
repo, state='open')` with the parameter names the service actually uses.

## Why the spec, and not a wrapper

Hand-written wrappers are better where they exist, and worse than nothing where they do not.
This is for everything else: the internal service with a `/openapi.json`, the API whose
client is three versions behind, the endpoint you need once. `fossick` finds the spec --
`apis` captures what a page calls, `fetch` reads a docs page -- and this turns it into
something callable.

## Why it is a Host and not a tool

Loading a spec is a session-scoped fact, like a vault or a kernel: the operations stay
available for the rest of the conversation, and the agent should be able to browse them
without re-fetching. Putting it on the host means one implementation for the CLI and for an
IDE that embeds this, which is what `VaultHost` established.


In [ ]:
#| export
import json
from pathlib import Path
from urllib.parse import urlparse

from ramabana.core import AgentError, agent_err
from ramabana.tools import LocalHost


In [ ]:
#| export
MAX_OPS = 400          # a spec larger than this is a catalogue, not a working surface


class SpecError(AgentError): "A specification could not be read, or an operation could not be called."


def load_spec(src, timeout=30):
    "A spec from a URL, a path, or an already-parsed dict, routed on what `src` is."
    if isinstance(src, dict): return src
    s = str(src or '').strip()
    if not s: raise SpecError('a spec url, path or dict is required')
    if urlparse(s).scheme in ('http', 'https'):
        import httpx2 as httpx
        try:
            r = httpx.get(s, timeout=timeout, follow_redirects=True); r.raise_for_status()
            return r.json()
        except Exception as e: raise SpecError(f'could not read the spec at {s}: {agent_err(e)}') from e
    p = Path(s).expanduser()
    if not p.is_file(): raise SpecError(f'no spec at {p}')
    try: return json.loads(p.read_text())
    except Exception as e: raise SpecError(f'{p} is not a JSON spec: {agent_err(e)}') from e


def spec_ops(spec):
    "Every operation in `spec` as `OpSpec` records, whatever flavour of spec it is."
    from fastcore.basics import AttrDict
    from fastspec.spec import openapi_to_ops
    # `openapi_to_ops` reads the document by attribute (`spec.paths`), not as a dict
    try: ops = list(openapi_to_ops(AttrDict(spec)))
    except Exception as e: raise SpecError(f'could not read the operations: {agent_err(e)}') from e
    if not ops: raise SpecError('the spec declares no operations')
    return ops[:MAX_OPS]


def op_row(op):
    "One operation as the shape a person or a model reads, signed by `fastcore.apisurface`."
    from fastcore.apisurface import mk_sig, sanitized_params
    try: sig = str(mk_sig(op, sanitized_params(_op_params(op)), op.param_defaults))
    except Exception: sig = '(...)'
    return dict(group=op.group or '', name=op.name, verb=(op.verb or '').upper(), path=op.path,
                summary=(op.summary or '').strip(), signature=f'{op.name}{sig}',
                required=list(op.required_params or []), docs_url=op.docs_url or '')


def _op_params(op):
    "Every parameter name an operation takes, in the order the signature wants them."
    return [*(op.route_params or []), *(op.query_params or []),
            *(op.body_params or []), *(op.file_params or [])]


In [ ]:
#| export
class SpecHost(LocalHost):
    """`LocalHost` that can also read an API specification and call what it declares.

    Everything `LocalHost` does is unchanged. What is added is one loaded spec per name, so a
    conversation can hold several -- an internal service and the vendor API it wraps -- and
    the agent names which one it means.

    Nothing is loaded at construction. A spec is a network fetch, and a host whose tool list
    waits on one is the mistake `VaultHost` documents at length.
    """

    @property
    def capabilities(self):
        "Declared rather than probed: the probe would be `api_ops`, which needs a loaded spec."
        return {**super().capabilities, 'api': True}

    def __init__(self, *a, specs=None, headers=None, timeout=60.0, **kw):
        super().__init__(*a, **kw)
        self.specs, self.headers, self.timeout = dict(specs or {}), dict(headers or {}), timeout
        self._clients = {}

    def api_load(self, src, name=''):
        "Read a spec and remember it under `name` (default: its title, else the host)."
        spec = load_spec(src)
        ops = spec_ops(spec)
        info = spec.get('info') or {}
        key = str(name or info.get('title') or urlparse(str(src)).netloc or 'api').strip()
        self.specs[key] = spec
        self._clients.pop(key, None)
        groups = sorted({o.group or '' for o in ops})
        return dict(name=key, title=info.get('title', ''), version=info.get('version', ''),
                    operations=len(ops), groups=groups)

    def api_names(self): return sorted(self.specs)

    def _spec(self, name=''):
        if not self.specs: raise SpecError('no specification loaded; call api_load first')
        if not name:
            if len(self.specs) > 1:
                raise SpecError(f'name which api: {", ".join(sorted(self.specs))}')
            name = next(iter(self.specs))
        if name not in self.specs: raise SpecError(f'no api {name!r}; loaded: {", ".join(sorted(self.specs))}')
        return name, self.specs[name]

    def api_ops(self, group='', name='', match=''):
        "Operations, optionally narrowed to one group or a substring of the name or summary."
        _, spec = self._spec(name)
        rows = [op_row(o) for o in spec_ops(spec)]
        if group: rows = [r for r in rows if r['group'] == group]
        if match:
            m = match.lower()
            rows = [r for r in rows if m in r['name'].lower() or m in r['summary'].lower()]
        return rows

    def api_call(self, operation, name='', **params):
        """Call one operation by name, with its own parameter names.

        Sync on purpose: a tool call is a blocking step in a turn, and an async client here
        would mean every caller managing a loop to get one response.
        """
        key, spec = self._spec(name)
        client = self._clients.get(key)
        if client is None:
            from fastspec.oapi import OpenAPIClient
            client = self._clients[key] = OpenAPIClient(spec, headers=self.headers,
                                                        timeout=self.timeout, sync=True)
        fn = getattr(client, str(operation), None)
        if fn is None:
            for grp in dir(client):
                sub = getattr(client, grp, None)
                if hasattr(sub, str(operation)): fn = getattr(sub, str(operation)); break
        if fn is None: raise SpecError(f'{key} declares no operation {operation!r}')
        try: return fn(**params)
        except Exception as e: raise SpecError(f'{operation} failed: {agent_err(e)}') from e


## Reading a spec

Everything below runs.

In [ ]:
SPEC = {'openapi': '3.0.0', 'info': {'title': 'Widgets', 'version': '1.2.0'},
        'paths': {'/widgets': {'get': {'operationId': 'listWidgets', 'tags': ['widgets'],
                                       'summary': 'Every widget',
                                       'parameters': [{'name': 'limit', 'in': 'query',
                                                       'schema': {'type': 'integer'}}]}},
                  '/widgets/{id}': {'get': {'operationId': 'getWidget', 'tags': ['widgets'],
                                            'summary': 'One widget',
                                            'parameters': [{'name': 'id', 'in': 'path',
                                                            'required': True,
                                                            'schema': {'type': 'string'}}]}}}}

h = SpecHost(roots=['.'])
loaded = h.api_load(SPEC, 'widgets')
test_eq(loaded['operations'], 2)
test_eq(loaded['title'], 'Widgets')

rows = {r['name']: r for r in h.api_ops(name='widgets')}
test_eq(sorted(rows), ['get_widget', 'list_widgets'])
# the signature carries the service's own parameter names, which is the whole point
assert 'id' in rows['get_widget']['signature'], rows['get_widget']['signature']
assert 'limit' in rows['list_widgets']['signature'], rows['list_widgets']['signature']
test_eq(rows['get_widget']['verb'], 'GET')
test_eq(rows['get_widget']['required'], ['id'])

# narrowing, because a real spec has hundreds of these
test_eq([r['name'] for r in h.api_ops(match='one widget', name='widgets')], ['get_widget'])
test_eq(h.api_ops(group='nothing', name='widgets'), [])

# naming which api is required once more than one is loaded
h.api_load(SPEC, 'other')
test_fail(lambda: h.api_ops(), contains='name which api')
test_fail(lambda: h.api_ops(name='nope'), contains='no api')
print(rows['get_widget']['signature'])


In [ ]:
# The group appears because the host declares `api`, and stays absent on a plain LocalHost.
from ramabana.tools import tools_for
names = {t.__name__ for t in tools_for(SpecHost(roots=['.']))}
assert {'api_load', 'api_ops', 'api_call'} <= names, sorted(names)
assert not {'api_load', 'api_ops', 'api_call'} & {t.__name__ for t in tools_for(LocalHost(roots=['.']))}
print(sorted(n for n in names if n.startswith('api_')))
